# Run a few points on your laptop

<div class='alert alert-info'>

In this how-to you set up a grid to run on your own computer, without
SLURM, with `backend = shell`. This is the quickest way to try a change
to the inlists or a Fortran source on one or two binaries before you run
a whole grid on a cluster.

</div>

<div class='alert alert-info'>

The outputs on this page were made with a stand-in for MESA, so the
compilation is instant, and the paths were shortened to `/home/user/...`.
With your own MESA, `posydon-grid setup` compiles MESA, which takes a few
minutes, and the paths are yours.

</div>

## Before you start

You need the same things as in
[Set up an HMS-HMS grid with a POSYDON rerun](hms_hms_rerun.ipynb#Before-you-start):

- POSYDON installed, with `posydon-grid` on your `PATH`;
- `MESA_DIR` and `MESASDK_ROOT` set, and the MESA SDK loaded, for the
  POSYDON version of MESA r11701;
- a clone of POSYDON-MESA-INLISTS on its `v2_sources` branch in
  `~/POSYDON-MESA-INLISTS`.

Make a new, empty folder for this grid and start Jupyter (or a terminal)
there. The cells below run in that folder; a line starting with `!` is a
shell command, which you can also type in a terminal without the `!`.

## 1. Pick the points

Keep it small: every point is a full MESA run, which takes from minutes
to hours. Two binaries:

In [1]:
import csv

with open("grid.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["initial_z", "Zbase", "m1", "m2",
                     "initial_period_in_days"])
    writer.writerow([0.0142, 0.0142, 30.0, 21.0, 10.0])
    writer.writerow([0.0142, 0.0142, 15.0, 10.5, 5.0])

## 2. The ini file

The only difference from a cluster grid is `[submission]`: `backend =
shell` writes a shell script that runs the points one after the other,
and `cpus_per_task` is the number of OpenMP threads of each MESA run (set
it to the number of cores you want to use). There are no SLURM settings.

In [2]:
%%writefile HMS-HMS_laptop.ini
[grid]
type = HMS-HMS
mode = fixed
points = grid.csv
run_directory = run
zams = ~/POSYDON-MESA-INLISTS/r11701/ZAMS_models/zams_z1.42m2_y0.2703.data

[scenario]
release = v2.0
inlist_repo = ~/POSYDON-MESA-INLISTS

[output]
history_interval = 1
star1 = history, final_model
star2 = history, final_model
binary = history

[submission]
backend = shell
job_array = true
cpus_per_task = 4

Writing HMS-HMS_laptop.ini


<details>
<summary><strong>job_array</strong></summary>

`job_array = true` runs the points one by one, as a SLURM job array
would. `job_array = false` writes a single `mpirun` instead; that is only
for dynamic grids, where psy-cris picks the points while the grid runs,
and is refused for a fixed grid.

</details>

## 3. Set up

In [3]:
!posydon-grid setup HMS-HMS_laptop.ini
!ls run

binary           grid_command.sh  setup_record.ini star2
column_lists     mk               star1


Instead of the SLURM scripts there is `grid_command.sh`. It loops over
the points (`0 1` here, the rows of `grid.csv`), runs each with
`posydon-run-grid`, and compresses the output at the end:

In [4]:
!head -8 run/grid_command.sh

#!/bin/bash

export OMP_NUM_THREADS=4

export MESASDK_ROOT=/home/user/mesasdk
source $MESASDK_ROOT/bin/mesasdk_init.sh
export MESA_DIR=/home/user/MESA/mesa-r11701



## 4. Run

Run it from the run directory, in a terminal where MESA is loaded:

```bash
cd run
./grid_command.sh
```

MESA's screen output does not appear in the terminal: each point runs in
its own folder in `run/`, named after its parameters, with MESA's output
in `out.txt` there. Follow a run with, for example,

```bash
tail -f run/Zbase_0.0142_m1_30.0000_*_grid_index_0/out.txt
```

<details>
<summary><strong>Running only one point</strong></summary>

Either keep only that row in `grid.csv` and run `posydon-grid setup`
again, or edit the loop in `grid_command.sh` (`for SLURM_ARRAY_TASK_ID in
0 1`) to the indices you want. The index is the row of `grid.csv`,
starting at 0.

</details>

<details>
<summary><strong>Setting up again</strong></summary>

After changing the `ini` file or an inlist, run `posydon-grid setup`
again. It replaces the files it writes (and warns about each), but
leaves other files, including the output of earlier runs, where they
are. For a clean start, remove the run directory first or use a new
`run_directory`.

</details>

## 5. Then on the cluster

When the test points look right, set up the whole grid on the cluster
with the same `ini` file, a full `grid.csv`, and `[submission]` changed to
`backend = slurm_array` with your SLURM settings, as in
[Set up an HMS-HMS grid with a POSYDON rerun](hms_hms_rerun.ipynb).